# 04 — Query ohsome API for the 10-cell test sample

This notebook performs the **first real ohsome API requests** for the deterministic 10-cell test sample prepared in Notebook 03.

The workflow is intentionally conservative:

1. query `/metadata` and record the current OSHDB temporal extent and API version;
2. load the 10-cell boundary sample prepared by Notebook 03;
3. build a compact `bpolys` GeoJSON FeatureCollection keyed by `grid_1km_id`;
4. query four grouped endpoints for the road network:
   - `/elements/length/groupBy/boundary`
   - `/elements/count/groupBy/boundary`
   - `/users/count/groupBy/boundary`
   - `/contributions/count/groupBy/boundary`
5. use the filter `highway=* and geometry:line`;
6. parse grouped responses to a tabular structure keyed by `grid_1km_id`;
7. persist raw JSON responses and a tidy Parquet summary for reproducibility.

This notebook queries **only 10 test cells**. It does not yet query all 895 analytical cells.

Official documentation:
- https://docs.ohsome.org/ohsome-api/v1/endpoints.html
- https://docs.ohsome.org/ohsome-api/stable/group-by.html


In [ ]:
from pathlib import Path
import json
import time
from datetime import datetime, timezone

import pandas as pd
import geopandas as gpd
import requests

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

API_BASE = "https://api.ohsome.org/v1"
ROAD_FILTER = "highway=* and geometry:line"
REQUEST_TIMEOUT = 120

print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("Requests:", requests.__version__)


## 1. Locate Notebook 03 outputs


In [ ]:
def find_project_root(start: Path) -> Path:
    """Find the repository root."""
    start = start.resolve()
    for candidate in [start, start.parent, *start.parents]:
        if (candidate / "notebooks").exists() and (candidate / "Petropolis").exists():
            return candidate
    raise FileNotFoundError("Could not locate the repository root.")

PROJECT_ROOT = find_project_root(Path.cwd())

INPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "intermediate"
    / "ohsome"
    / "Petropolis"
)

TEST_BOUNDARIES_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_boundaries.geojson"
TEST_INDEX_FILE = INPUT_DIR / "petropolis_grid_1km_ohsome_test_cells.parquet"

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "ohsome"
    / "Petropolis"
    / "test_10_cells"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Petropolis"
)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

SUMMARY_FILE = RESULTS_DIR / "petropolis_ohsome_test_10_cells.parquet"
METADATA_FILE = OUTPUT_DIR / "ohsome_metadata.json"

for path in [TEST_BOUNDARIES_FILE, TEST_INDEX_FILE]:
    if not path.exists():
        raise FileNotFoundError(
            f"Required Notebook 03 output not found: {path}. "
            "Execute Notebook 03 before Notebook 04."
        )

print("Project root:", PROJECT_ROOT)
print("Test boundaries:", TEST_BOUNDARIES_FILE)
print("Test index:", TEST_INDEX_FILE)
print("Raw output directory:", OUTPUT_DIR)


## 2. Query `/metadata`

The metadata endpoint is queried before analytical requests. The complete response is stored verbatim so that the API version and the temporal extent available in the backing OSHDB are documented for this run.


In [ ]:
session = requests.Session()
session.headers.update({
    "User-Agent": "sotm-br-2026-research/0.1 (academic OSM analysis)"
})

metadata_url = f"{API_BASE}/metadata"
metadata_response = session.get(metadata_url, timeout=REQUEST_TIMEOUT)
metadata_response.raise_for_status()
metadata = metadata_response.json()

with METADATA_FILE.open("w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print("HTTP status:", metadata_response.status_code)
print("Metadata saved to:", METADATA_FILE)
display(pd.json_normalize(metadata))


## 3. Extract and record the OSHDB temporal extent


In [ ]:
def recursive_find(obj, key_fragments):
    """Find scalar values whose key contains any requested fragment."""
    found = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            key_lower = str(key).lower()
            if any(fragment.lower() in key_lower for fragment in key_fragments):
                if not isinstance(value, (dict, list)):
                    found.append((key, value))
            found.extend(recursive_find(value, key_fragments))
    elif isinstance(obj, list):
        for item in obj:
            found.extend(recursive_find(item, key_fragments))
    return found

temporal_candidates = recursive_find(
    metadata,
    ["timestamp", "temporal", "time", "start", "end", "from", "to"]
)

api_version = (
    metadata.get("apiVersion")
    or metadata.get("api_version")
    or metadata.get("version")
)

print("API version:", api_version)
print("\nTemporal metadata candidates:")
for key, value in temporal_candidates:
    print(f"  {key}: {value}")

RUN_UTC = datetime.now(timezone.utc).isoformat()
print("\nNotebook run time (UTC):", RUN_UTC)


## 4. Determine the query time

For this first test, element length and element count are evaluated at the **latest snapshot available in OSHDB** by omitting the `time` parameter, which is the documented default behavior for element aggregations.

For users and contributions, the notebook uses the full available temporal history when the metadata response exposes a usable start/end range. If the metadata structure changes and the range cannot be extracted automatically, execution stops rather than silently inventing a temporal interval.


In [ ]:
def parse_iso_candidate(value):
    if not isinstance(value, str):
        return None
    text = value.strip()
    if len(text) < 10:
        return None
    try:
        return pd.Timestamp(text)
    except Exception:
        return None

parsed_temporal = []
for key, value in temporal_candidates:
    ts = parse_iso_candidate(value)
    if ts is not None:
        parsed_temporal.append((key, ts))

if not parsed_temporal:
    raise ValueError(
        "Could not automatically identify temporal timestamps in /metadata. "
        "Inspect the metadata response before running history-based requests."
    )

timestamps = [ts for _, ts in parsed_temporal]
oshdb_start = min(timestamps)
oshdb_end = max(timestamps)

if oshdb_start == oshdb_end:
    raise ValueError(
        "Only one temporal timestamp was detected in metadata; "
        "a valid historical interval cannot be inferred safely."
    )

history_time = f"{oshdb_start.isoformat()},{oshdb_end.isoformat()}"

print("Inferred OSHDB start:", oshdb_start.isoformat())
print("Inferred OSHDB end:", oshdb_end.isoformat())
print("History interval:", history_time)


## 5. Load and validate the 10-cell test sample


In [ ]:
test_cells = gpd.read_parquet(TEST_INDEX_FILE)

with TEST_BOUNDARIES_FILE.open("r", encoding="utf-8") as f:
    test_feature_collection = json.load(f)

feature_ids = [
    str(feature.get("properties", {}).get("id"))
    for feature in test_feature_collection.get("features", [])
]

checks = {
    "test_cell_count_is_10": len(test_cells) == 10,
    "grid_id_unique": test_cells["grid_1km_id"].is_unique,
    "grid_id_non_null": test_cells["grid_1km_id"].notna().all(),
    "feature_count_is_10": len(feature_ids) == 10,
    "feature_ids_unique": len(feature_ids) == len(set(feature_ids)),
    "feature_ids_match_index": set(feature_ids) == set(test_cells["grid_1km_id"].astype(str)),
}

display(pd.Series(checks, name="PASS"))

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError(f"Test-boundary validation failed: {failed}")

display(
    test_cells[
        ["grid_1km_id", "spatial_relation", "municipality_fraction", "source_method"]
    ].sort_values("grid_1km_id")
)


## 6. Prepare the `bpolys` payload


In [ ]:
bpolys = json.dumps(
    test_feature_collection,
    ensure_ascii=False,
    separators=(",", ":")
)

payload_size_kib = len(bpolys.encode("utf-8")) / 1024

print("Boundary count:", len(feature_ids))
print(f"bpolys payload size: {payload_size_kib:.2f} KiB")


## 7. Define a robust ohsome POST helper

Every successful response is written to disk before parsing. Retries are limited and used only for transient server-side errors or request exceptions.


In [ ]:
def ohsome_post(endpoint, data, raw_filename, max_attempts=3, pause_seconds=5):
    """POST to ohsome API with limited retry logic and raw-response persistence."""
    url = f"{API_BASE}{endpoint}"
    last_exception = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = session.post(
                url,
                data=data,
                timeout=REQUEST_TIMEOUT,
            )

            if response.status_code >= 500 and attempt < max_attempts:
                print(
                    f"Transient HTTP {response.status_code} for {endpoint}; "
                    f"retry {attempt}/{max_attempts}."
                )
                time.sleep(pause_seconds)
                continue

            response.raise_for_status()
            payload = response.json()

            raw_path = OUTPUT_DIR / raw_filename
            with raw_path.open("w", encoding="utf-8") as f:
                json.dump(payload, f, ensure_ascii=False, indent=2)

            print(f"PASS — {endpoint} → HTTP {response.status_code}")
            print("Saved:", raw_path)
            return payload

        except (requests.RequestException, ValueError) as exc:
            last_exception = exc
            if attempt < max_attempts:
                print(
                    f"Request error for {endpoint}: {exc}. "
                    f"Retry {attempt}/{max_attempts}."
                )
                time.sleep(pause_seconds)
            else:
                break

    raise RuntimeError(
        f"ohsome request failed after {max_attempts} attempts: {endpoint}"
    ) from last_exception


## 8. Query current road length by `grid_1km_id`


In [ ]:
length_payload = ohsome_post(
    endpoint="/elements/length/groupBy/boundary",
    data={
        "bpolys": bpolys,
        "filter": ROAD_FILTER,
        "format": "json",
    },
    raw_filename="elements_length_groupby_boundary.json",
)


## 9. Query current road-element count by `grid_1km_id`


In [ ]:
element_count_payload = ohsome_post(
    endpoint="/elements/count/groupBy/boundary",
    data={
        "bpolys": bpolys,
        "filter": ROAD_FILTER,
        "format": "json",
    },
    raw_filename="elements_count_groupby_boundary.json",
)


## 10. Query unique contributors over the available OSHDB history


In [ ]:
users_payload = ohsome_post(
    endpoint="/users/count/groupBy/boundary",
    data={
        "bpolys": bpolys,
        "filter": ROAD_FILTER,
        "time": history_time,
        "format": "json",
    },
    raw_filename="users_count_groupby_boundary.json",
)


## 11. Query contributions over the available OSHDB history

The contributions endpoint is documented by ohsome as experimental. Its raw response is therefore retained unchanged in addition to the parsed table.


In [ ]:
contributions_payload = ohsome_post(
    endpoint="/contributions/count/groupBy/boundary",
    data={
        "bpolys": bpolys,
        "filter": ROAD_FILTER,
        "time": history_time,
        "format": "json",
    },
    raw_filename="contributions_count_groupby_boundary.json",
)


## 12. Inspect grouped-response structures


In [ ]:
for name, payload in {
    "length": length_payload,
    "element_count": element_count_payload,
    "users": users_payload,
    "contributions": contributions_payload,
}.items():
    print(f"\n--- {name} ---")
    print("Top-level keys:", list(payload.keys()))
    groups = payload.get("groupByResult", [])
    print("Groups:", len(groups))
    if groups:
        print("First group keys:", list(groups[0].keys()))
        print(json.dumps(groups[0], ensure_ascii=False, indent=2)[:2000])


## 13. Parse grouped scalar results

ohsome aggregation responses generally store one grouped object per boundary and a `result` list containing one or more timestamped values. This parser extracts the latest scalar value for each boundary while keeping the function strict enough to fail visibly if the response structure changes.


In [ ]:
def extract_group_scalar(payload, value_name):
    rows = []

    groups = payload.get("groupByResult")
    if not isinstance(groups, list):
        raise ValueError("Expected 'groupByResult' list in ohsome response.")

    for group in groups:
        grid_id = str(group.get("groupByObject"))
        result = group.get("result")

        if not isinstance(result, list) or not result:
            rows.append({
                "grid_1km_id": grid_id,
                value_name: pd.NA,
                f"{value_name}_timestamp": pd.NaT,
            })
            continue

        last_item = result[-1]
        if not isinstance(last_item, dict):
            raise ValueError(
                f"Unexpected result item for boundary {grid_id}: {type(last_item)}"
            )

        candidate_value_keys = [
            key for key in last_item.keys()
            if key not in {"timestamp", "fromTimestamp", "toTimestamp"}
        ]

        if len(candidate_value_keys) != 1:
            raise ValueError(
                f"Could not uniquely determine scalar value key for boundary {grid_id}: "
                f"{candidate_value_keys}"
            )

        value_key = candidate_value_keys[0]

        rows.append({
            "grid_1km_id": grid_id,
            value_name: last_item.get(value_key),
            f"{value_name}_timestamp": last_item.get(
                "timestamp",
                last_item.get("toTimestamp")
            ),
        })

    return pd.DataFrame(rows)

length_df = extract_group_scalar(length_payload, "road_length_m")
element_count_df = extract_group_scalar(element_count_payload, "road_element_count")
users_df = extract_group_scalar(users_payload, "contributors_count")
contributions_df = extract_group_scalar(contributions_payload, "contributions_count")

display(length_df.head())
display(element_count_df.head())
display(users_df.head())
display(contributions_df.head())


## 14. Join all test metrics by the permanent grid key


In [ ]:
base_columns = [
    "grid_1km_id",
    "spatial_relation",
    "municipality_fraction",
    "cell_area_m2",
    "intersection_area_m2",
    "source_method",
]

summary = test_cells[base_columns].copy()
summary["grid_1km_id"] = summary["grid_1km_id"].astype(str)

for metrics in [
    length_df,
    element_count_df,
    users_df,
    contributions_df,
]:
    summary = summary.merge(
        metrics,
        on="grid_1km_id",
        how="left",
        validate="one_to_one",
    )

summary["road_length_km"] = pd.to_numeric(
    summary["road_length_m"], errors="coerce"
) / 1000.0

summary["road_density_km_per_km2"] = (
    summary["road_length_km"]
    / (pd.to_numeric(summary["cell_area_m2"], errors="coerce") / 1_000_000.0)
)

summary["oshdb_start"] = oshdb_start.isoformat()
summary["oshdb_end"] = oshdb_end.isoformat()
summary["ohsome_api_version"] = api_version
summary["query_filter"] = ROAD_FILTER
summary["query_run_utc"] = RUN_UTC

display(summary.sort_values("grid_1km_id"))


## 15. Quality control of API results


In [ ]:
metric_columns = [
    "road_length_m",
    "road_element_count",
    "contributors_count",
    "contributions_count",
]

qc_rows = []

for column in metric_columns:
    numeric = pd.to_numeric(summary[column], errors="coerce")
    qc_rows.append({
        "metric": column,
        "non_null": int(numeric.notna().sum()),
        "null": int(numeric.isna().sum()),
        "negative": int((numeric < 0).sum()),
        "zero": int((numeric == 0).sum()),
        "min": numeric.min(),
        "max": numeric.max(),
    })

qc = pd.DataFrame(qc_rows)
display(qc)

checks = {
    "summary_has_10_rows": len(summary) == 10,
    "summary_grid_id_unique": summary["grid_1km_id"].is_unique,
    "all_test_ids_retained": set(summary["grid_1km_id"]) == set(feature_ids),
    "no_negative_road_length": (
        pd.to_numeric(summary["road_length_m"], errors="coerce").dropna() >= 0
    ).all(),
    "no_negative_element_count": (
        pd.to_numeric(summary["road_element_count"], errors="coerce").dropna() >= 0
    ).all(),
    "no_negative_contributor_count": (
        pd.to_numeric(summary["contributors_count"], errors="coerce").dropna() >= 0
    ).all(),
    "no_negative_contribution_count": (
        pd.to_numeric(summary["contributions_count"], errors="coerce").dropna() >= 0
    ).all(),
}

display(pd.Series(checks, name="PASS"))

if not all(checks.values()):
    failed = [name for name, passed in checks.items() if not passed]
    raise ValueError(f"API result quality control failed: {failed}")


## 16. Save the tidy 10-cell test result


In [ ]:
summary.to_parquet(SUMMARY_FILE, index=False)

print("Saved:", SUMMARY_FILE)
print("Rows:", len(summary))
print("Columns:", len(summary.columns))


## 17. Interpretation and next step

If all checks pass, the 10-cell test confirms that the spatial boundary representation, permanent `grid_1km_id`, road filter, and four ohsome aggregation endpoints can be integrated into one reproducible table.

The next notebook should **not immediately send all 895 cells in one request**. Instead it should:

1. define a conservative batching strategy;
2. reuse the exact same boundary IDs and filter;
3. persist every raw API response;
4. add retry/backoff and resumable cache logic;
5. expand the temporal design for snapshots and contribution intervals;
6. only then scale to all selected 1 km cells.
